## Deep Control Variates by Variance Minimization

This notebook implements the deep learning control variance method using variance minimization as proposed by [Vidales et al. 2018](https://arxiv.org/pdf/1810.05094v1).

In [ ]:
import QuantLib as ql
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import math
import numpy as np
from numpy.linalg import cholesky, norm
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Basket Option Parameters

In [ ]:
dim = 10
S0 = 1.0
r = 0.05
div_yield = 0.03
sigma = 0.2
T = 1.0
K = 1.0
correlations = np.identity(dim)

In [ ]:
spot_prices = [S0 for i in range(dim)]
dividend_yields = [div_yield for i in range(dim)]
volatilities = [sigma for i in range(dim)]

### Quantlib Basket Option

In [ ]:
def price_basket_option(dim, spot_prices, volatilities, correlations, strike, maturity, 
                        risk_free_rate, dividend_yields, option_type='call'):
    # Convert input parameters to QuantLib types
    calendar = ql.TARGET()
    day_count = ql.Actual365Fixed()
    settlement_date = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = settlement_date
    maturity_date = settlement_date + ql.Period(int(maturity * 365), ql.Days)
    risk_free_curve = ql.YieldTermStructureHandle(ql.FlatForward(settlement_date, ql.QuoteHandle(ql.SimpleQuote(risk_free_rate)), day_count))
    dividend_curves = [ql.YieldTermStructureHandle(ql.FlatForward(settlement_date, ql.QuoteHandle(ql.SimpleQuote(div)), day_count)) for div in dividend_yields]
    spot_handles = [ql.QuoteHandle(ql.SimpleQuote(s)) for s in spot_prices]
    volatility_handles = [ql.BlackVolTermStructureHandle(ql.BlackConstantVol(settlement_date, calendar, ql.QuoteHandle(ql.SimpleQuote(vol)), day_count)) for vol in volatilities]
    
    # Correlation matrix
    correlation_matrix = ql.Matrix(dim, dim)
    for i in range(dim):
        for j in range(dim):
            correlation_matrix[i][j] = correlations[i][j]
    
    # Create basket of assets
    processes = []
    for i in range(dim):
        process = ql.BlackScholesMertonProcess(spot_handles[i], dividend_curves[i], 
                                               risk_free_curve, volatility_handles[i])
        processes.append(process)
    
    basket_process = ql.StochasticProcessArray(processes, correlation_matrix)
    
    # Create the payoff and the basket option
    if option_type.lower() == 'call':
        payoff = ql.PlainVanillaPayoff(ql.Option.Call, strike)
    else:
        payoff = ql.PlainVanillaPayoff(ql.Option.Put, strike)
    basket_payoff = ql.AverageBasketPayoff(payoff, dim)
    exercise = ql.EuropeanExercise(maturity_date)
    basket_option = ql.BasketOption(basket_payoff, exercise)
    
    # Set up the pricing engine
    engine = ql.MCEuropeanBasketEngine(basket_process, 'pseudorandom', timeStepsPerYear=1, requiredSamples=5000)
    basket_option.setPricingEngine(engine)
    
    # Calculate the price
    price = basket_option.NPV()
    error_estimate = basket_option.errorEstimate()
    
    return price, error_estimate

In [ ]:
start_time = time.time()
option_price, error_est = price_basket_option(dim, spot_prices, volatilities, 
                                   correlations, K, T, r, dividend_yields)
total_time = time.time() - start_time
print(f"Basket Option Price: {option_price:.4f}")
print(f"Error estimate: {error_est:.8f}")

In [ ]:
print(f"QL Basket (CPU) Execution Time: {total_time} seconds")

### PyTorch Monte Carlo

In [ ]:
def monte_carlo_basket_option_price(dim, spot_prices, volatilities, correlations, strike, 
                                    maturity, risk_free_rate, dividend_yields, 
                                    num_simulations=10000, num_time_steps=1, option_type='call', device=device):
    # Convert inputs to tensors and move to the specified device
    spot_prices = torch.tensor(spot_prices, device=device, dtype=torch.float32)
    volatilities = torch.tensor(volatilities, device=device, dtype=torch.float32)
    correlations = torch.tensor(correlations, device=device, dtype=torch.float32)
    dividend_yields = torch.tensor(dividend_yields, device=device, dtype=torch.float32)
    
    dt = maturity / num_time_steps
    sqrtdt = np.sqrt(dt)
    discount_factor = np.exp(-risk_free_rate * maturity)
    
    # Cholesky decomposition for the correlation matrix
    chol_matrix = torch.linalg.cholesky(correlations)
    
    # Simulate asset paths
    def simulate_paths():
        Z = torch.randn(num_time_steps, dim, num_simulations, device=device)
        transformed_samples = torch.matmul(chol_matrix, Z)
        
        dS = (risk_free_rate - dividend_yields.unsqueeze(1) - 0.5 * volatilities.unsqueeze(1)**2) * dt + \
             volatilities.unsqueeze(1) * sqrtdt * transformed_samples
        cum_sum = torch.cumsum(dS, dim=0)
        exp_S = torch.exp(cum_sum)
        S = spot_prices.unsqueeze(1) * exp_S
        return S
    
    paths = simulate_paths()
    
    # Calculate the basket average at maturity
    basket_avg = paths[-1].mean(dim=0)
    
    # Calculate payoff
    if option_type.lower() == 'call':
        payoffs = torch.maximum(basket_avg - strike, torch.tensor(0.0, device=device))
    else:
        payoffs = torch.maximum(strike - basket_avg, torch.tensor(0.0, device=device))
    
    # Calculate discounted payoff
    discounted_payoffs = discount_factor * payoffs
    price = discounted_payoffs.mean().item()
    std_error = discounted_payoffs.std().item() / np.sqrt(num_simulations)
    
    return price, std_error

In [ ]:
no_paths = 5000
num_time_steps = 1

In [ ]:
start_time = time.time()
option_price, standard_error = monte_carlo_basket_option_price(dim, spot_prices, volatilities, correlations, 
                                                               K, T, r, dividend_yields, no_paths, 
                                                               num_time_steps, 'call', device)
total_time = time.time() - start_time
print(f"Basket Option Price: {option_price:.4f}")
print(f"Standard Error: {standard_error:.8f}")

In [ ]:
print(f"MC Basket (GPU) Execution Time: {total_time} seconds")

### Deep Importance Sampling

In [ ]:
class DeepISNetwork(nn.Module):
    def __init__(self, input_size, width):
        super(DeepISNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, input_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
class DeepControlVariate(nn.Module):
    def __init__(self, dim, spot_prices, volatilities, correlations, strike, 
                 maturity, risk_free_rate, dividend_yields, num_simulations=10000, 
                 num_time_steps=1, option_type='call', device=device):
        super(DeepControlVariate, self).__init__()
        self.dim = dim
        self.volatilities = torch.tensor(volatilities, device=device, dtype=torch.float32)
        self.strike = strike
        self.maturity = maturity
        self.risk_free_rate = risk_free_rate
        self.dividend_yields = torch.tensor(dividend_yields, device=device, dtype=torch.float32)
        self.num_simulations = num_simulations
        self.num_time_steps = num_time_steps
        self.option_type = option_type
        self.device = device
        correlations = torch.tensor(correlations, device=device, dtype=torch.float32)
        self.cholesky =  torch.linalg.cholesky(correlations)
        self.df = np.exp(-self.risk_free_rate * self.maturity)
        self.network_lst = nn.ModuleList([DeepISNetwork(dim, 64) for i in range(num_time_steps)])  

    def forward(self, S0):
        spot_prices = torch.tensor(S0, device=self.device, dtype=torch.float32)
        dt = self.maturity / self.num_time_steps
        sqrtdt = np.sqrt(dt)
        
        Z = torch.randn(self.num_time_steps, self.dim, self.num_simulations, device=self.device)
        transformed_samples = torch.matmul(self.cholesky, Z)
        vol_sqrt_dW = self.volatilities.unsqueeze(1) * sqrtdt * transformed_samples
        
        # Calculate path increments and path
        dS = (self.risk_free_rate - self.dividend_yields.unsqueeze(1) - \
              0.5 * self.volatilities.unsqueeze(1)**2) * dt + \
              vol_sqrt_dW
        cum_sum = torch.cumsum(dS, dim=0)
        exp_S = torch.exp(cum_sum)
        S = spot_prices.unsqueeze(1) * exp_S
        
        # Calculate martingale control variate
        netZ_int_lst = list()
        for i, net in enumerate(self.network_lst):
            netZ_int_lst.append(net(dS[i].T))
        net_Z = torch.cat(netZ_int_lst)
        stoch_int = torch.matmul(net_Z, vol_sqrt_dW)
        control_variate = torch.cumsum(stoch_int, dim=0)
        
        return S, control_variate

In [ ]:
def train_optimise_var(model, spot_prices, n_iter, optimizer, base_lr, device=device):
    model.train()
    tqdm_epoch = trange(n_iter)
    
    best_loss = float('inf')
    epochs_no_improve = 0
    
    history = {
        'loss': list(),
        'MC_Price': list(),
        'MC_std_error': list()
    }
    
    for it in tqdm_epoch:
        model.train()
        optimizer.zero_grad()
        
        lr = base_lr * 0.1**(it//5000)
        for param_group in optimizer.state_dict()['param_groups']:
            param_group['lr'] = lr
        
        # Forward pass
        paths, control_variate = model(spot_prices)
        basket_avg = paths[-1].mean(dim=0)
    
        # Calculate payoff
        if model.option_type.lower() == 'call':
            payoffs = torch.maximum(basket_avg - model.strike, torch.tensor(0.0, device=device))
        else:
            payoffs = torch.maximum(model.strike - basket_avg, torch.tensor(0.0, device=device))
    
        discounted_payoffs = model.df * payoffs
        price = discounted_payoffs.mean().item()
        std_error = discounted_payoffs.std().item() / np.sqrt(model.num_simulations)
        
        MC_CV = price - model.df * control_variate
        MC_Price = MC_CV.mean()
        var_MC_CV_estimator = MC_CV.var()
        loss = var_MC_CV_estimator
        loss.backward()
        
        optimizer.step()
        MC_std_error = MC_CV.std().item() / np.sqrt(model.num_simulations)
        
        # Update progress bar
        tqdm_epoch.set_description(f"Iteration {it+1}/{n_iter} \
            - Loss: {loss:.8e}, Price: {price:.4f}, Std Error: {std_error:.8e} \
            MC Price: {MC_Price:.4f}, Std Error: {MC_std_error:.8e}, lr:{lr:.2e}")
        
        # Store metrics
        history['loss'].append(loss.item())
        history['MC_Price'].append(MC_Price.item())
        history['MC_std_error'].append(MC_std_error)
    
    return history

In [ ]:
model = DeepControlVariate(dim, spot_prices, volatilities, correlations, 
                           K, T, r, dividend_yields, 5000, 
                           num_time_steps, 'call', device).to(device)

In [ ]:
num_epochs = 20000    # Number of epochs for training
learning_rate = 0.0001 # Learning rate

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
history = train_optimise_var(model, spot_prices, num_epochs, optimizer, learning_rate, device=device)

In [ ]:
epochs = range(0, num_epochs)
plt.figure(figsize=(12,6)) 
    
loss_values = history['loss']    
plt.plot(epochs, loss_values, color='k', linestyle='-')

plt.xlabel('Iterations')
plt.ylabel('MSE Loss')
plt.yscale('log')
plt.savefig('DeepCVLoss.png', dpi=300, bbox_inches='tight')

In [ ]:
var_red_values = standard_error / history['MC_std_error']

In [ ]:
plt.figure(figsize=(12,6)) 
    

plt.plot(epochs, var_red_values, color='k', linestyle='-')
plt.ylabel('Standard Error Reduction')

plt.savefig('DeepCVVarRed.png', dpi=300, bbox_inches='tight')

In [ ]:
batches = 1000

start_time = time.time()

results = list()
for i in range(batches):
    option_price2, _ = monte_carlo_basket_option_price(dim, spot_prices, volatilities, correlations, 
                                                               K, T, r, dividend_yields, 10000000, 
                                                               num_time_steps, 'call', device)
    results.append(option_price2)

option_price2 = np.mean(results)
standard_error2 = np.std(results) / np.sqrt(batches)
total_time = time.time() - start_time
print(f"Basket Option Price: {option_price2:.4f}")
print(f"Standard Error: {standard_error2:.8e}")

In [ ]:
print(f"MC Basket (GPU) Execution Time: {total_time} seconds")